# Select Valid Subjects (T1, T2 and BOLD in the Same Session) and Keep the First Valid Session

In [21]:
# Import necessary libraries:
import os
import re
import shutil
from datetime import datetime

# Create dry-run variable to control directory deleting:
DRY_RUN = False

# Create logs directory if it does not exist:
os.makedirs("logs", exist_ok=True)

# Create output folder:
out_dir = "logs/1.valid_subjects_sessions"
os.makedirs(out_dir, exist_ok=True)

# Create unique timestamp for this run:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create unique log filenames for this run:
kept_subjects_log = f"logs/1.valid_subjects_sessions/kept_subjects_log_{timestamp}.txt"
deleted_subjects_log = f"logs/1.valid_subjects_sessions/deleted_subjects_log_{timestamp}.txt"

kept_sessions_log = f"logs/1.valid_subjects_sessions/kept_sessions_log_{timestamp}.txt"
deleted_sessions_log = f"logs/1.valid_subjects_sessions/deleted_sessions_log_{timestamp}.txt"

summary_file = f"logs/1.valid_subjects_sessions/summary_report_{timestamp}.txt"

# Define root directory of the dataset:
root_dir = "../../dataset/raw"

# Helper function to extract numeric part from ses-dXXXX and return integer:
def get_session_number(session_name):
    match = re.search(r'ses-d(\d+)', session_name)
    return int(match.group(1))

# Helper function to check if anat folder contains at least one T1 and one T2 file:
def anat_has_T1_T2(anat_path, strict=True):
    # Return false if the anat folder does not exist:
    if not os.path.isdir(anat_path):
        return False

    # Make a list of all files within the anat folder:
    files = os.listdir(anat_path)

    # Check if any filename contains "T1":
    if strict:
        has_T1 = any(("T1" in f) and ("echo" not in f.lower()) and ("hippocampus" not in f.lower()) for f in files)
    else:              
        has_T1 = any(("T1" in f) and ("hippocampus" not in f.lower()) for f in files)

    # Check if any filename contains "T2":
    if strict:
        has_T2 = any(("T2" in f) and ("TSE" not in f) and ("echo" not in f.lower()) for f in files)
    else:
        has_T2 = any("T2" in f for f in files)

    # Return true only if both T1 and T2 files exist, otherwise false:
    return has_T1 and has_T2

# Create counters for summary:
kept_subjects = 0
deleted_subjects = 0

# Iterate over all subjects in the dataset:
for sub in sorted(os.listdir(root_dir)):

    # Extract the path of the current subject:
    sub_path = os.path.join(root_dir, sub)

    # Skip if not a directory and move to the next loop iteration:
    if not os.path.isdir(sub_path):
        continue

    # Find sessions of the current subject directory:
    sessions = [s for s in os.listdir(sub_path) if os.path.isdir(os.path.join(sub_path, s))]

    # Sort the found sessions by days of entry on the study:
    sessions_sorted = sorted(sessions, key=get_session_number)

    # Create a variable to store the first valid session for the current subject:
    valid_session = None

    # Iterate over all the sessions of the current subject:

    # FIRST PASS, STRICT: Search for valid session for the current subject including a T1 structural file and a T2 structural file without TSE #
    for ses in sessions_sorted:
        # Create the full path of the current session of the current subject:
        ses_path = os.path.join(sub_path, ses)

        # Create both the anat and func folder paths of the current session of the current subject:
        anat_path = os.path.join(ses_path, "anat")
        func_path = os.path.join(ses_path, "func")

        # If the func folder is missing, the current session is skipped:
        if not os.path.isdir(func_path):
            continue

        # Call a helper function anat_has_T1_T2 with the strict parameter:
        if anat_has_T1_T2(anat_path, strict=True):
            # If a valid session has been found for the current subject, exit the session loop:
            valid_session = ses
            break

    # SECOND PASS, RELAXED: If no valid session was found for the current subject in the previous step, allow any T2 structural file including TSE types #
    if valid_session is None:
        for ses in sessions_sorted:
            # Create the full path of the current session of the current subject:
            ses_path = os.path.join(sub_path, ses)

            # Create both the anat and func folder paths of the current session of the current subject:
            anat_path = os.path.join(ses_path, "anat")
            func_path = os.path.join(ses_path, "func")

            # If the func folder is missing, the current session is skipped:
            if not os.path.isdir(func_path):
                continue

            # Call a helper function anat_has_T1_T2:
            if anat_has_T1_T2(anat_path, strict=False):
                valid_session = ses
                break

    # If no valid session exists for the subject, we delete the entire subject folder:
    if valid_session is None:
        # Show the message of deleting subject:
        message = f"Deleting subject (no valid session): {sub}"
        print(message)

        # Save the deleted subject in the log file:
        with open(deleted_subjects_log, "a") as log:
            log.write(sub + "\n")
            
        # Delete the entire subject folder is the dry run is disabled:
        if not DRY_RUN:
            shutil.rmtree(sub_path)

        # Increment the deleted subjects counter:
        deleted_subjects += 1
        continue

    # Keep only the first valid session and delete the rest:
    print(f"Keeping {sub}/{valid_session} and deleting the rest.")

    # Save kept subject in the log file:
    with open(kept_subjects_log, "a") as log:
        log.write(sub + "\n")

    # Save kept session in the log file:
    with open(kept_sessions_log, "a") as log:
        log.write(f"{sub}/{valid_session}\n")

    # Iterate over the sessions of the current subject and delete all but the valid one:
    for ses in sessions:
        if ses != valid_session:
            # Build the path of the current session to delete:
            ses_path = os.path.join(sub_path, ses)
            if os.path.isdir(ses_path):
                # Print the message of the current session being deleted:
                message = f"Deleting sessions: {sub}/{ses}"
                print(message)

                # Save the deleted session in the log file:
                with open(deleted_sessions_log, "a") as log:
                    log.write(f"{sub}/{ses}\n")
                    
                # Delete the session folder is the dry run is disabled:
                if not DRY_RUN:
                    shutil.rmtree(ses_path)
                    
    # Increment the kept subjects counter:
    kept_subjects += 1

# Prepare the summary text:
summary_text = (
    "| SUMMARY |\n"
    f"Total subjects processed: {kept_subjects + deleted_subjects}\n"
    f"Subjects kept: {kept_subjects}\n"
    f"Subjects deleted: {deleted_subjects}\n"
)

# Print to console:
print(summary_text)

# Save to file:
with open(summary_file, "w") as f:
    f.write(summary_text)

Keeping sub-OAS30001/ses-d0129 and deleting the rest.
Deleting sessions: sub-OAS30001/ses-d2430
Deleting sessions: sub-OAS30001/ses-d3746
Deleting sessions: sub-OAS30001/ses-d3132
Deleting sessions: sub-OAS30001/ses-d0757
Deleting sessions: sub-OAS30001/ses-d4467
Keeping sub-OAS30002/ses-d0653 and deleting the rest.
Deleting sessions: sub-OAS30002/ses-d1680
Deleting sessions: sub-OAS30002/ses-d2345
Deleting sessions: sub-OAS30002/ses-d0371
Deleting sessions: sub-OAS30002/ses-d2340
Keeping sub-OAS30003/ses-d0558 and deleting the rest.
Deleting sessions: sub-OAS30003/ses-d4954
Deleting sessions: sub-OAS30003/ses-d2682
Deleting sessions: sub-OAS30003/ses-d1631
Deleting sessions: sub-OAS30003/ses-d2669
Deleting sessions: sub-OAS30003/ses-d3731
Deleting sessions: sub-OAS30003/ses-d3320
Keeping sub-OAS30004/ses-d1101 and deleting the rest.
Deleting sessions: sub-OAS30004/ses-d4700
Deleting sessions: sub-OAS30004/ses-d3457
Deleting sessions: sub-OAS30004/ses-d2232
Deleting sessions: sub-OAS30